# Cloud Cost Optimization, Part 3: Interactive dashboard

This notebook builds the interactive dashboard for the project, a single web page saved as `docs/index.html` and published with GitHub Pages: [gowdilyan.github.io/cloud_cost_optimization](https://gowdilyan.github.io/cloud_cost_optimization/).

The page has six interactive charts made with Plotly:
1. Daily spend, as a total or split by environment or provider
2. Forecast vs actual for each month from April to December
3. The spend alert, with the 95% expected range and the flagged days
4. A heatmap of every model's error in every month
5. A savings calculator for the recommendations
6. The grid search results for Random Forest and Gradient Boosting

The models are the same as in [Part 2](2_cloud_cost_modeling.ipynb), with the same settings and the same backtest. Part 2 only kept the scores, so here I run the backtest again and keep the daily forecasts for the charts. The scores come out the same as in Part 2. This notebook takes about 3 minutes to run.

In [1]:
import itertools
import logging
import os
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.forecasting.stl import STLForecast
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import mean_absolute_percentage_error
from xgboost import XGBRegressor
from prophet import Prophet

warnings.filterwarnings('ignore')
logging.getLogger('cmdstanpy').disabled = True    # Prophet prints a lot of progress messages otherwise

## 1. Load the data

In [2]:
df = pd.read_csv('data/cloud_budget_2023_dataset.csv', parse_dates=['date'])

daily = df.groupby('date')['amortized_cost'].sum().asfreq('D')
env_daily = df.pivot_table(index='date', columns='environment', values='amortized_cost', aggfunc='sum')[['prod', 'staging', 'dev']]
provider_daily = df.pivot_table(index='date', columns='cloud_provider', values='amortized_cost', aggfunc='sum')
print(len(daily), 'days')

365 days


## 2. The models from Part 2

These are copied from Part 2 without changes: the two baselines, the order searches for the time series models, the features and grid searches for the regression models, and the Prophet tuning.

In [3]:
def get_mape(actual, pred):
    return round(mean_absolute_percentage_error(actual, pred) * 100, 2)


def naive_forecast(history, days):
    return np.repeat(history.iloc[-1], days)


def seasonal_naive_forecast(history, days):
    last_week = history.iloc[-7:].values
    return np.tile(last_week, days // 7 + 1)[:days]


def search_arima(history):
    results = []
    for p, q in itertools.product([0, 1, 2], [0, 1, 2]):
        model = ARIMA(history, order=(p, 1, q)).fit()
        results.append({'order': (p, 1, q), 'AIC': round(model.aic, 1)})
    return pd.DataFrame(results).sort_values('AIC').reset_index(drop=True)


def search_sarima(history):
    results = []
    for p, q, P, Q in itertools.product([0, 1, 2], [0, 1, 2], [0, 1], [0, 1]):
        model = SARIMAX(history, order=(p, 1, q), seasonal_order=(P, 1, Q, 7),
                        enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
        results.append({'order': (p, 1, q), 'seasonal_order': (P, 1, Q, 7), 'AIC': round(model.aic, 1)})
    return pd.DataFrame(results).sort_values('AIC').reset_index(drop=True)


def search_stl(history):
    results = []
    for p, q in itertools.product([0, 1, 2], [0, 1, 2]):
        for trend in ['n', 't']:
            model = STLForecast(history, ARIMA, model_kwargs={'order': (p, 1, q), 'trend': trend}, period=7).fit()
            results.append({'order': (p, 1, q), 'trend': trend, 'AIC': round(model.model_result.aic, 1)})
    return pd.DataFrame(results).sort_values('AIC').reset_index(drop=True)


def make_features(series):
    features = pd.DataFrame(index=series.index)
    features['day_of_week'] = features.index.dayofweek
    features['is_weekend'] = (features['day_of_week'] >= 5).astype(int)
    features['month'] = features.index.month
    features['day'] = features.index.day
    features['lag_35'] = series.shift(35)
    features['lag_42'] = series.shift(42)
    features['rolling_35'] = series.shift(35).rolling(7).mean()
    return features


features = make_features(daily).dropna()
features_linear = pd.get_dummies(features, columns=['day_of_week'], drop_first=True, dtype=int)

tscv = TimeSeriesSplit(n_splits=5)
scoring = 'neg_mean_absolute_percentage_error'
ridge_grid = GridSearchCV(Pipeline([('scale', StandardScaler()), ('ridge', Ridge())]),
                          param_grid={'ridge__alpha': [0.1, 1, 10, 100, 1000]}, cv=tscv, scoring=scoring)
rf_grid = GridSearchCV(RandomForestRegressor(random_state=42),
                       param_grid={'n_estimators': [100, 300], 'max_depth': [2, 3, 5, None], 'min_samples_leaf': [1, 5, 10]},
                       cv=tscv, scoring=scoring)
gb_grid = GridSearchCV(GradientBoostingRegressor(random_state=42),
                       param_grid={'n_estimators': [50, 100, 300], 'max_depth': [1, 2, 3], 'learning_rate': [0.01, 0.05, 0.1]},
                       cv=tscv, scoring=scoring)
xgb_grid = GridSearchCV(XGBRegressor(random_state=42, n_jobs=1),
                        param_grid={'n_estimators': [100, 300], 'max_depth': [1, 2, 3],
                                    'learning_rate': [0.01, 0.05, 0.1], 'subsample': [0.8, 1.0]},
                        cv=tscv, scoring=scoring)

holidays = pd.to_datetime(['2023-01-02', '2023-01-16', '2023-02-20', '2023-05-29',
                           '2023-07-04', '2023-09-04', '2023-11-23', '2023-12-25'])
prophet_holidays = pd.DataFrame({'holiday': 'us_holiday', 'ds': holidays})


def prophet_forecast(history, days, cps):
    model = Prophet(weekly_seasonality=True, yearly_seasonality=False, daily_seasonality=False,
                    changepoint_prior_scale=cps, holidays=prophet_holidays)
    model.fit(pd.DataFrame({'ds': history.index, 'y': history.values}))
    future = model.make_future_dataframe(periods=days)
    return model.predict(future)['yhat'].values[-days:]


def tune_prophet(history):
    fit_part, holdout = history[:-30], history[-30:]
    scores = {}
    for cps in [0.01, 0.05, 0.1, 0.5]:
        scores[cps] = mean_absolute_percentage_error(holdout, prophet_forecast(fit_part, 30, cps))
    return min(scores, key=scores.get)

## 3. Backtest with the daily forecasts

Same as the backtest in Part 2: for each month from April to December, every model is tuned and trained only on the days before the month. This time I keep each model's daily forecast, and the SARIMA 95% range for the alert chart.

In [4]:
forecasts = {}      # (month, model) -> daily forecast
alert_bands = {}    # month -> SARIMA forecast, 95% range and actual

for month_start in pd.date_range('2023-04-01', '2023-12-01', freq='MS'):
    month_end = month_start + pd.offsets.MonthEnd(0)
    last_day = month_start - pd.Timedelta(days=1)
    history = daily[:last_day]
    actual = daily[month_start:month_end]
    days = len(actual)
    month = month_start.strftime('%b')

    a_order = search_arima(history).loc[0, 'order']
    s_best = search_sarima(history).loc[0]
    stl_best = search_stl(history).loc[0]

    sarima = SARIMAX(history, order=s_best['order'], seasonal_order=s_best['seasonal_order'],
                     enforce_stationarity=False, enforce_invertibility=False).fit(disp=False).get_forecast(days)
    band = sarima.conf_int()
    band.columns = ['lower', 'upper']
    band['forecast'] = sarima.predicted_mean
    band['actual'] = actual
    alert_bands[month] = band

    y_hist = daily[features[:last_day].index]
    month_preds = {
        'Naive': naive_forecast(history, days),
        'Seasonal naive': seasonal_naive_forecast(history, days),
        'ARIMA': ARIMA(history, order=a_order).fit().forecast(days),
        'SARIMA': sarima.predicted_mean,
        'STL + ARIMA': STLForecast(history, ARIMA, model_kwargs={'order': stl_best['order'], 'trend': stl_best['trend']},
                                   period=7).fit().forecast(days),
        'Ridge': clone(ridge_grid).fit(features_linear[:last_day], y_hist).predict(features_linear[month_start:month_end]),
        'Random Forest': clone(rf_grid).fit(features[:last_day], y_hist).predict(features[month_start:month_end]),
        'Gradient Boosting': clone(gb_grid).fit(features[:last_day], y_hist).predict(features[month_start:month_end]),
        'XGBoost': clone(xgb_grid).fit(features[:last_day], y_hist).predict(features[month_start:month_end]),
        'Prophet': prophet_forecast(history, days, tune_prophet(history))
    }
    for name, pred in month_preds.items():
        forecasts[(month, name)] = pd.Series(np.asarray(pred), index=actual.index)

month_order = ['Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
normal_months = ['Apr', 'May', 'Jul', 'Sep', 'Oct', 'Dec']
shift_months = ['Jun', 'Aug', 'Nov']
model_names = list(month_preds.keys())

mape_table = pd.DataFrame({month: {name: get_mape(alert_bands[month]['actual'], forecasts[(month, name)])
                                   for name in model_names} for month in month_order})
mape_table['normal months'] = mape_table[normal_months].mean(axis=1).round(2)
mape_table['shift months'] = mape_table[shift_months].mean(axis=1).round(2)
mape_table['all 9 months'] = mape_table[month_order].mean(axis=1).round(2)
mape_table = mape_table.sort_values('normal months')
mape_table[['normal months', 'shift months', 'all 9 months']]

,normal months,shift months,all 9 months
SARIMA,5.65,17.41,9.57
Random Forest,5.97,15.59,9.18
STL + ARIMA,6.02,16.95,9.66
Prophet,6.03,17.59,9.88
XGBoost,6.15,14.60,8.96
Gradient Boosting,6.29,14.65,9.08
Seasonal naive,7.08,16.90,10.36
Ridge,7.84,16.00,10.56
ARIMA,11.19,18.10,13.49
Naive,14.97,20.23,16.72


These match the backtest tables in Part 2 (sections 10 and 12).

The alert chart uses the same four months as Part 2: the two normal months October and December, and the two shift months June and November.

In [5]:
alert_months = ['Jun', 'Oct', 'Nov', 'Dec']
pd.DataFrame({month: {'days above range': int((alert_bands[month]['actual'] > alert_bands[month]['upper']).sum()),
                      'days in month': len(alert_bands[month])} for month in alert_months}).T

,days above range,days in month
Jun,16,30
Oct,0,31
Nov,24,30
Dec,0,31


## 4. Grid search results

The grid search explorer on the page uses the same grid searches as Part 2, trained on January to November.

In [6]:
X_train = features[:'2023-11-30']
y_train = daily[X_train.index]
rf_grid.fit(X_train, y_train)
gb_grid.fit(X_train, y_train)

rf_cv = pd.DataFrame(rf_grid.cv_results_)
rf_cv['CV MAPE %'] = (-rf_cv['mean_test_score'] * 100).round(2)
rf_cv['param_max_depth'] = rf_cv['param_max_depth'].fillna('None').astype(str)

gb_cv = pd.DataFrame(gb_grid.cv_results_)
gb_cv['CV MAPE %'] = (-gb_cv['mean_test_score'] * 100).round(2)

print('Random Forest best:', rf_grid.best_params_, round(-rf_grid.best_score_ * 100, 2))
print('Gradient Boosting best:', gb_grid.best_params_, round(-gb_grid.best_score_ * 100, 2))

Random Forest best: {'max_depth': 5, 'min_samples_leaf': 10, 'n_estimators': 100} 9.14
Gradient Boosting best: {'learning_rate': 0.01, 'max_depth': 1, 'n_estimators': 300} 8.8


## 5. The charts

Each chart is a Plotly figure. Dropdown menus switch between views, and the legend can show or hide lines. I don't display the charts in the notebook, since GitHub doesn't show interactive Plotly output. They're saved to the web page in the last step.

In [7]:
# Chart 1: daily spend, as a total or split by environment or provider
fig_daily = go.Figure()
fig_daily.add_trace(go.Scatter(x=daily.index, y=daily.round(0), name='Total', line=dict(color='black', width=1.5),
                               hovertemplate='%{x|%a %b %d}: $%{y:,.0f}<extra></extra>'))
for col in env_daily.columns:
    fig_daily.add_trace(go.Scatter(x=env_daily.index, y=env_daily[col].round(0), name=col, stackgroup='env', visible=False,
                                   hovertemplate=col + ' %{x|%a %b %d}: $%{y:,.0f}<extra></extra>'))
for col in provider_daily.columns:
    fig_daily.add_trace(go.Scatter(x=provider_daily.index, y=provider_daily[col].round(0), name=col, stackgroup='provider', visible=False,
                                   hovertemplate=col + ' %{x|%a %b %d}: $%{y:,.0f}<extra></extra>'))

n_env = len(env_daily.columns)
n_provider = len(provider_daily.columns)
weekend_shading = [dict(type='rect', xref='x', yref='paper', x0=saturday, x1=saturday + pd.Timedelta(days=2), y0=0, y1=1,
                        fillcolor='lightgrey', opacity=0.35, line_width=0, layer='below')
                   for saturday in daily.index[daily.index.dayofweek == 5]]

fig_daily.update_layout(
    title='Daily cloud spend in 2023 (grey bands are weekends)',
    yaxis_title='USD per day', height=500, shapes=weekend_shading,
    xaxis=dict(rangeslider=dict(visible=True)),
    updatemenus=[dict(type='buttons', direction='right', x=0, y=1.12, xanchor='left', buttons=[
        dict(label='Total', method='update', args=[{'visible': [True] + [False] * (n_env + n_provider)}]),
        dict(label='By environment', method='update', args=[{'visible': [False] + [True] * n_env + [False] * n_provider}]),
        dict(label='By provider', method='update', args=[{'visible': [False] + [False] * n_env + [True] * n_provider}])
    ])]
)
print('daily spend chart:', len(fig_daily.data), 'lines')

daily spend chart: 7 lines


In [8]:
# Chart 2: forecast vs actual, one month at a time
shown_by_default = ['SARIMA', 'STL + ARIMA', 'Gradient Boosting']
palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
model_colors = dict(zip(model_names, palette))    # same color for a model in every month
fig_forecast = go.Figure()
trace_months = []

for month in month_order:
    actual = alert_bands[month]['actual']
    fig_forecast.add_trace(go.Scatter(x=actual.index, y=actual.round(0), name='Actual', visible=(month == 'Dec'),
                                      line=dict(color='black', width=2.5), mode='lines+markers', marker=dict(size=4),
                                      hovertemplate='Actual %{x|%a %b %d}: $%{y:,.0f}<extra></extra>'))
    trace_months.append((month, 'Actual'))
    for name in model_names:
        mape = mape_table.loc[name, month]
        visible = (True if name in shown_by_default else 'legendonly') if month == 'Dec' else False
        fig_forecast.add_trace(go.Scatter(x=actual.index, y=forecasts[(month, name)].round(0), name=f'{name} ({mape:.1f}%)',
                                          visible=visible, line=dict(color=model_colors[name]),
                                          hovertemplate=name + ' %{x|%a %b %d}: $%{y:,.0f}<extra></extra>'))
        trace_months.append((month, name))


def month_visibility(selected):
    return [(True if name in shown_by_default + ['Actual'] else 'legendonly') if month == selected else False
            for month, name in trace_months]


fig_forecast.update_layout(
    title='Forecast vs actual: Dec (the % is each model\'s MAPE for the month)',
    yaxis_title='USD per day', height=560, margin=dict(t=120), title_y=0.97,
    updatemenus=[dict(type='dropdown', x=0, y=1.14, xanchor='left', active=month_order.index('Dec'), buttons=[
        dict(label=month, method='update',
             args=[{'visible': month_visibility(month)},
                   {'title.text': f'Forecast vs actual: {month} (the % is each model\'s MAPE for the month)'}])
        for month in month_order
    ])]
)
print('forecast chart:', len(fig_forecast.data), 'lines (actual + 10 models for each of the 9 months)')

forecast chart: 99 lines (actual + 10 models for each of the 9 months)


In [9]:
# Chart 3: the spend alert
fig_alert = go.Figure()
alert_traces = []

for month in alert_months:
    band = alert_bands[month]
    flagged = band[band['actual'] > band['upper']]
    visible = month == 'Nov'
    fig_alert.add_trace(go.Scatter(x=band.index, y=band['upper'].round(0), line=dict(width=0), showlegend=False,
                                   visible=visible, hoverinfo='skip'))
    fig_alert.add_trace(go.Scatter(x=band.index, y=band['lower'].round(0), line=dict(width=0), fill='tonexty',
                                   fillcolor='rgba(214, 39, 40, 0.15)', name='Expected range (95%)', visible=visible,
                                   customdata=band['upper'].round(0),
                                   hovertemplate='Expected range %{x|%b %d}: $%{y:,.0f} to $%{customdata:,.0f}<extra></extra>'))
    fig_alert.add_trace(go.Scatter(x=band.index, y=band['forecast'].round(0), name='SARIMA forecast', visible=visible,
                                   line=dict(color='rgb(214, 39, 40)'),
                                   hovertemplate='Forecast %{x|%b %d}: $%{y:,.0f}<extra></extra>'))
    fig_alert.add_trace(go.Scatter(x=band.index, y=band['actual'].round(0), name='Actual', visible=visible,
                                   line=dict(color='black'), mode='lines+markers', marker=dict(size=4),
                                   hovertemplate='Actual %{x|%a %b %d}: $%{y:,.0f}<extra></extra>'))
    fig_alert.add_trace(go.Scatter(x=flagged.index, y=flagged['actual'].round(0), name='Alert: above expected range',
                                   visible=visible, mode='markers', marker=dict(color='orange', size=11, line=dict(color='black', width=1)),
                                   hovertemplate='ALERT %{x|%a %b %d}: $%{y:,.0f}<extra></extra>'))
    alert_traces += [month] * 5


def alert_title(month):
    band = alert_bands[month]
    above = int((band['actual'] > band['upper']).sum())
    return f'{month}: forecast made the day before the month starts. {above} of {len(band)} days above the expected range'


fig_alert.update_layout(
    title=alert_title('Nov'), title_y=0.97, yaxis_title='USD per day', height=560, margin=dict(t=120),
    updatemenus=[dict(type='dropdown', x=0, y=1.14, xanchor='left', active=alert_months.index('Nov'), buttons=[
        dict(label=month, method='update',
             args=[{'visible': [m == month for m in alert_traces]}, {'title.text': alert_title(month)}])
        for month in alert_months
    ])]
)
print('alert chart:', len(fig_alert.data), 'traces (5 for each of the 4 months)')

alert chart: 20 traces (5 for each of the 4 months)


In [10]:
# Chart 4: heatmap of every model's MAPE in every month
heat = mape_table[month_order + ['normal months', 'shift months', 'all 9 months']]
fig_heat = go.Figure(go.Heatmap(
    z=heat.values, x=heat.columns, y=heat.index, texttemplate='%{z:.1f}',
    colorscale='RdYlGn_r', colorbar=dict(title='MAPE %'),
    hovertemplate='%{y}, %{x}: %{z:.2f}% MAPE<extra></extra>'))
fig_heat.update_layout(title='MAPE % by model and month (lower is better, sorted by the normal months)',
                       height=480, yaxis=dict(autorange='reversed'))
print('heatmap:', heat.shape[0], 'models x', heat.shape[1], 'columns')

heatmap: 10 models x 12 columns


In [11]:
# Chart 6: grid search explorer, one figure per model with a dropdown for the third setting
def grid_figure(cv, row_param, col_param, menu_param, title, row_label, col_label, menu_label):
    fig = go.Figure()
    menu_values = sorted(cv[menu_param].unique())
    for value in menu_values:
        table = cv[cv[menu_param] == value].pivot(index=row_param, columns=col_param, values='CV MAPE %')
        fig.add_trace(go.Heatmap(z=table.values, x=[str(c) for c in table.columns], y=[str(r) for r in table.index],
                                 text=table.values, texttemplate='%{text:.2f}', colorscale='Blues_r',
                                 zmin=cv['CV MAPE %'].min(), zmax=cv['CV MAPE %'].max(), visible=(value == menu_values[0]),
                                 colorbar=dict(title='CV MAPE %'),
                                 hovertemplate=row_label + ' %{y}, ' + col_label + ' %{x}: %{z:.2f}%<extra></extra>'))
    fig.update_layout(
        title=title, height=420, xaxis_title=col_label, yaxis_title=row_label,
        xaxis=dict(type='category'), yaxis=dict(type='category'),
        updatemenus=[dict(type='dropdown', x=0, y=1.2, xanchor='left', buttons=[
            dict(label=f'{menu_label} = {value}', method='update',
                 args=[{'visible': [v == value for v in menu_values]}])
            for value in menu_values
        ])]
    )
    return fig


fig_grid_rf = grid_figure(rf_cv, 'param_max_depth', 'param_min_samples_leaf', 'param_n_estimators',
                          'Random Forest: CV MAPE % for each setting', 'max_depth', 'min_samples_leaf', 'n_estimators')
fig_grid_gb = grid_figure(gb_cv, 'param_learning_rate', 'param_n_estimators', 'param_max_depth',
                          'Gradient Boosting: CV MAPE % for each setting', 'learning_rate', 'n_estimators', 'max_depth')

The savings calculator (chart 5) isn't a Plotly chart. It's two sliders with a few lines of JavaScript, using the yearly totals below. These are the same numbers as the recommendations in Part 2.

In [12]:
weekend = df[df['date'].dt.dayofweek >= 5]
weekend_by_env = weekend.groupby('environment')['amortized_cost'].sum()
total_by_env = df.groupby('environment')['amortized_cost'].sum()

savings_numbers = {
    'dev_weekend': round(weekend_by_env['dev']),
    'staging_weekend': round(weekend_by_env['staging']),
    'prod_total': round(total_by_env['prod']),
    'total': round(df['amortized_cost'].sum())
}
savings_numbers

{'dev_weekend': 5329,
 'staging_weekend': 13697,
 'prod_total': 273867,
 'total': 354376}

## 6. Build the web page

The page loads Plotly from its CDN, so the HTML file only holds the data and the layout. After this runs, `docs/index.html` can be opened in any browser. To publish it, turn on GitHub Pages in the repository settings (Settings, then Pages, then deploy from the `main` branch and the `/docs` folder).

In [13]:
def chart_html(fig, first=False):
    return fig.to_html(full_html=False, include_plotlyjs='cdn' if first else False, config={'displaylogo': False})


savings_html = """
<div class="calc">
  <label for="nonprod">Share of dev and staging weekend spend that is scaled down: <b><span id="nonprod_val"></span>%</b></label>
  <input type="range" id="nonprod" min="0" max="100" step="5" value="100">
  <label for="prod">Savings from a cost review of prod: <b><span id="prod_val"></span>%</b></label>
  <input type="range" id="prod" min="0" max="20" step="1" value="5">
  <p id="calc_result"></p>
</div>
<script>
const DEV_WEEKEND = __DEV__, STAGING_WEEKEND = __STAGING__, PROD_TOTAL = __PROD__, TOTAL = __TOTAL__;
function money(x) { return '$' + Math.round(x).toLocaleString('en-US'); }
function updateSavings() {
  const nonprod = Number(document.getElementById('nonprod').value);
  const prod = Number(document.getElementById('prod').value);
  document.getElementById('nonprod_val').textContent = nonprod;
  document.getElementById('prod_val').textContent = prod;
  const weekendSaving = (DEV_WEEKEND + STAGING_WEEKEND) * nonprod / 100;
  const prodSaving = PROD_TOTAL * prod / 100;
  const total = weekendSaving + prodSaving;
  document.getElementById('calc_result').innerHTML =
    'Weekend scale-down: <b>' + money(weekendSaving) + '</b> a year. ' +
    'Prod review: <b>' + money(prodSaving) + '</b> a year. ' +
    'Together: <b>' + money(total) + '</b> a year, about <b>' + (total / TOTAL * 100).toFixed(1) + '%</b> of the yearly bill.';
}
document.getElementById('nonprod').addEventListener('input', updateSavings);
document.getElementById('prod').addEventListener('input', updateSavings);
updateSavings();
</script>
"""
savings_html = (savings_html.replace('__DEV__', str(savings_numbers['dev_weekend']))
                .replace('__STAGING__', str(savings_numbers['staging_weekend']))
                .replace('__PROD__', str(savings_numbers['prod_total']))
                .replace('__TOTAL__', str(savings_numbers['total'])))

page = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Cloud Cost Forecasting Dashboard</title>
<style>
  body { font-family: -apple-system, Helvetica, Arial, sans-serif; max-width: 1100px; margin: 0 auto; padding: 20px; color: #222; line-height: 1.5; }
  h1 { margin-bottom: 0.2em; }
  h2 { margin-top: 2em; border-bottom: 1px solid #ddd; padding-bottom: 0.2em; }
  .calc { background: #f6f6f6; padding: 16px 20px; border-radius: 6px; }
  .calc label { display: block; margin-top: 10px; }
  .calc input { width: 100%; max-width: 500px; }
  footer { margin-top: 3em; font-size: 0.9em; color: #666; }
</style>
</head>
<body>
<h1>Cloud Cost Forecasting Dashboard</h1>
<p>This page goes with the <a href="https://github.com/gowdilyan/cloud_cost_optimization">cloud cost optimization project</a>.
It uses one year (2023) of daily cloud billing data to forecast next month's spend and to flag days when spend goes above the expected range.
All the charts are interactive: hover for exact values, drag to zoom, double-click to reset, and click a legend item to show or hide it.</p>
<p>Contents: <a href="#daily">1. Daily spend</a> | <a href="#forecast">2. Forecast vs actual</a> | <a href="#alert">3. Spend alert</a> |
<a href="#models">4. Model comparison</a> | <a href="#savings">5. Savings calculator</a> | <a href="#grid">6. Grid search</a></p>

<h2 id="daily">1. Daily spend</h2>
<p>Weekends (the grey bands) are about 24% cheaper than weekdays. Spend doesn't grow smoothly. It jumps in steps: up 16% in June, down 10% in August and up 29% in November.
Use the buttons to split the spend by environment or provider, and the slider under the chart to zoom into a period.</p>
__DAILY__

<h2 id="forecast">2. Forecast vs actual</h2>
<p>Each month was forecast using only the data before it. Pick a month from the dropdown. SARIMA, STL + ARIMA and Gradient Boosting are shown first, and the other models can be turned on in the legend.
In normal months the best models are within about 6% per day. In June, August and November, when spend jumped, every model missed.</p>
__FORECAST__

<h2 id="alert">3. Spend alert</h2>
<p>SARIMA, the final model, gives a 95% expected range for each day. A day above the range is flagged as an alert.
In November and June the alert went off on the first day of the month, and in the normal months (October and December) there were no alerts. Pick a month from the dropdown.</p>
__ALERT__

<h2 id="models">4. Model comparison</h2>
<p>Every model's MAPE (average daily error in %) for every month, with the averages for the normal months, the shift months and all nine months on the right. Hover over a cell for the exact value.</p>
__HEAT__

<h2 id="savings">5. Savings calculator</h2>
<p>Two of the recommendations are to scale down dev and staging on weekends and to focus cost reviews on prod.
Move the sliders to see what they could save in a year. Scaling down 100% of the weekend spend is the maximum, since some non-prod work probably does need to run on weekends.</p>
__SAVINGS__

<h2 id="grid">6. Grid search</h2>
<p>The cross-validation MAPE for every combination of settings tried for Random Forest and Gradient Boosting, using January to November only. Use the dropdown to change the third setting.</p>
__GRID_RF__
__GRID_GB__

<footer>Built with Plotly from <a href="https://github.com/gowdilyan/cloud_cost_optimization/blob/main/3_dashboard.ipynb">3_dashboard.ipynb</a>.
Data: <a href="https://www.kaggle.com/datasets/rasikaekanayakadevlk/cloud-budget-dataset">Cloud Budget Dataset on Kaggle</a> (synthetic).</footer>
</body>
</html>
"""

page = (page.replace('__DAILY__', chart_html(fig_daily, first=True))
        .replace('__FORECAST__', chart_html(fig_forecast))
        .replace('__ALERT__', chart_html(fig_alert))
        .replace('__HEAT__', chart_html(fig_heat))
        .replace('__SAVINGS__', savings_html)
        .replace('__GRID_RF__', chart_html(fig_grid_rf))
        .replace('__GRID_GB__', chart_html(fig_grid_gb)))

os.makedirs('docs', exist_ok=True)
with open('docs/index.html', 'w') as f:
    f.write(page)
print('saved docs/index.html,', round(os.path.getsize('docs/index.html') / 1024), 'KB')

saved docs/index.html, 337 KB
